In [127]:
!pip install scikit-learn==1.2.2 pyarrow tqdm matplotlib paramiko imbalanced-learn umap-learn xgboost lightgbm

In [128]:
import requests
import os
import time

# Create a directory for storage if it doesn't exist
os.makedirs('hapmap_data', exist_ok=True)

# List of required file URLs
file_urls = {
    'hapmap3_r3_b36_fwd.consensus.qc.poly.map.gz': 'https://ftp.ncbi.nlm.nih.gov/hapmap/genotypes/hapmap3_r3/plink_format/hapmap3_r3_b36_fwd.consensus.qc.poly.map.gz',
    'hapmap3_r3_b36_fwd.qc.poly.tar.gz': 'https://ftp.ncbi.nlm.nih.gov/hapmap/genotypes/hapmap3_r3/plink_format/hapmap3_r3_b36_fwd.qc.poly.tar.gz',
    'relationships_w_pops_041510.txt': 'https://ftp.ncbi.nlm.nih.gov/hapmap/genotypes/hapmap3_r3/relationships_w_pops_041510.txt'
}

# Download each file
for filename, url in file_urls.items():
    print(f"Starting download: {filename}")
    
    try:
        # Define the file save path
        save_path = os.path.join('hapmap_data', filename)
        
        # Make the HTTP request with progress display
        response = requests.get(url, stream=True)
        response.raise_for_status()  # Check for successful request
        
        # Get total file size if available
        total_size = int(response.headers.get('content-length', 0))
        total_size_mb = total_size / (1024 * 1024)
        
        print(f"File size: {total_size_mb:.2f} MB")
        
        # Save the file in chunks
        with open(save_path, 'wb') as file:
            downloaded = 0
            start_time = time.time()
            
            for chunk in response.iter_content(chunk_size=1024*1024):  # 1MB chunks
                if chunk:
                    file.write(chunk)
                    downloaded += len(chunk)
                    
                    # Show progress every 5MB
                    if downloaded % (5*1024*1024) == 0:
                        elapsed = time.time() - start_time
                        speed = downloaded / (1024 * 1024 * elapsed) if elapsed > 0 else 0
                        percent = (downloaded / total_size * 100) if total_size > 0 else 0
                        print(f"Downloaded: {downloaded/(1024*1024):.2f} MB ({percent:.1f}%) at {speed:.2f} MB/s")
        
        print(f"Successfully downloaded: {filename}")
        
    except Exception as e:
        print(f"Error downloading {filename}: {str(e)}")
    
    print("-" * 50)

print("All files have been downloaded successfully")

# Additional code to extract compressed files
import gzip
import tarfile

# Extract gzip files
def extract_gzip(gz_file, output_file):
    with gzip.open(gz_file, 'rb') as f_in:
        with open(output_file, 'wb') as f_out:
            f_out.write(f_in.read())
    print(f"Extracted: {gz_file} to {output_file}")

# Extract tar.gz files
def extract_tar_gz(tar_gz_file, output_dir):
    with tarfile.open(tar_gz_file, 'r:gz') as tar:
        tar.extractall(path=output_dir)
    print(f"Extracted: {tar_gz_file} to folder {output_dir}")

# Extract files after download
try:
    # Extract map gzip file
    map_gz = os.path.join('hapmap_data', 'hapmap3_r3_b36_fwd.consensus.qc.poly.map.gz')
    map_file = os.path.join('hapmap_data', 'hapmap3_r3_b36_fwd.consensus.qc.poly.map')
    if os.path.exists(map_gz):
        extract_gzip(map_gz, map_file)
    
    # Extract tar.gz file
    tar_gz_file = os.path.join('hapmap_data', 'hapmap3_r3_b36_fwd.qc.poly.tar.gz')
    if os.path.exists(tar_gz_file):
        extract_tar_gz(tar_gz_file, 'hapmap_data')
    
    print("All files have been extracted successfully")
except Exception as e:
    print(f"Error during extraction: {str(e)}")


Starting download: hapmap3_r3_b36_fwd.consensus.qc.poly.map.gz
File size: 10.79 MB
Downloaded: 5.00 MB (46.3%) at 7.02 MB/s
Downloaded: 10.00 MB (92.6%) at 11.29 MB/s
Successfully downloaded: hapmap3_r3_b36_fwd.consensus.qc.poly.map.gz
--------------------------------------------------
Starting download: hapmap3_r3_b36_fwd.qc.poly.tar.gz
File size: 1331.78 MB
Downloaded: 5.00 MB (0.4%) at 7.58 MB/s
Downloaded: 10.00 MB (0.8%) at 11.96 MB/s
Downloaded: 15.00 MB (1.1%) at 16.11 MB/s
Downloaded: 20.00 MB (1.5%) at 18.06 MB/s
Downloaded: 25.00 MB (1.9%) at 19.50 MB/s
Downloaded: 30.00 MB (2.3%) at 21.82 MB/s
Downloaded: 35.00 MB (2.6%) at 23.66 MB/s
Downloaded: 40.00 MB (3.0%) at 24.19 MB/s
Downloaded: 45.00 MB (3.4%) at 24.64 MB/s
Downloaded: 50.00 MB (3.8%) at 26.02 MB/s
Downloaded: 55.00 MB (4.1%) at 26.31 MB/s
Downloaded: 60.00 MB (4.5%) at 27.39 MB/s
Downloaded: 65.00 MB (4.9%) at 27.57 MB/s
Downloaded: 70.00 MB (5.3%) at 28.53 MB/s
Downloaded: 75.00 MB (5.6%) at 29.38 MB/s
Downloaded

In [129]:
import pandas as pd
import numpy as np
import os
import glob
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, roc_auc_score
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
import gc
import warnings
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d import Axes3D
from sklearn.metrics import roc_curve, auc, confusion_matrix
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')

In [130]:
data_dir = 'hapmap_data'
map_file = os.path.join(data_dir, 'hapmap3_r3_b36_fwd.consensus.qc.poly.map')
relationships_file = os.path.join(data_dir, 'relationships_w_pops_041510.txt')
output_dir = os.path.join(data_dir, 'sex_prediction_data')
os.makedirs(output_dir, exist_ok=True)

ped_files = glob.glob(os.path.join(data_dir, '**/*.ped'), recursive=True)
if not ped_files:
    raise FileNotFoundError("No PED files found. Make sure the archive was extracted correctly.")

print(f"Found {len(ped_files)} PED files for different populations")

Found 11 PED files for different populations


In [131]:
print("\nReading relationships and populations file...")
relationships_df = pd.read_csv(relationships_file, sep='\s+', low_memory=False)
print(f"Relationships file read. Dimensions: {relationships_df.shape}")

print("\nReading map file (SNPs information)...")
map_columns = ['CHR', 'SNP', 'GEN_DIST', 'POS']
map_df = pd.read_csv(map_file, sep='\s+', header=None, names=map_columns)
print(f"Map file read. Number of SNPs: {map_df.shape[0]}")

MAX_SNPS = 300000
if map_df.shape[0] > MAX_SNPS:
    print(f"\nUsing first {MAX_SNPS} SNPs out of {map_df.shape[0]} to speed up processing...")
    map_df = map_df.iloc[:MAX_SNPS]
    print(f"Number of SNPs limited. New SNP count: {map_df.shape[0]}")



Reading relationships and populations file...
Relationships file read. Dimensions: (1397, 7)

Reading map file (SNPs information)...
Map file read. Number of SNPs: 1457897

Using first 300000 SNPs out of 1457897 to speed up processing...
Number of SNPs limited. New SNP count: 300000


In [132]:
all_samples_info = []
population_labels = {}
sex_labels = {}

for ped_file in ped_files:
    file_basename = os.path.basename(ped_file)
    population = file_basename.split('.')[1] if len(file_basename.split('.')) > 2 else "Unknown"
    
    ped_columns = ['FID', 'IID', 'PAT', 'MAT', 'SEX', 'PHENO']
    dtype_dict = {0: str, 1: str, 2: str, 3: str, 4: int, 5: int}
    
    ped_info_df = pd.read_csv(ped_file, sep='\s+', header=None, usecols=range(6), 
                             names=ped_columns, dtype=dtype_dict, low_memory=False)
    
    ped_info_df['FILE_POP'] = population
    
    all_samples_info.append(ped_info_df)
    
    for idx, row in ped_info_df.iterrows():
        population_labels[row['IID']] = population
        sex_labels[row['IID']] = row['SEX']

all_samples_df = pd.concat(all_samples_info, ignore_index=True)
print(f"Collected information for {all_samples_df.shape[0]} samples from {len(ped_files)} populations.")


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
if 'SEX' not in all_samples_df.columns or all_samples_df['SEX'].isnull().all():
    raise ValueError("No sex data in PED files!")

sex_counts = all_samples_df['SEX'].value_counts()
print("\nSex distribution in the data:")
print(sex_counts)

valid_sex_mask = all_samples_df['SEX'].isin([1, 2])  # 1=male, 2=female
all_samples_df = all_samples_df[valid_sex_mask]
print(f"Data filtered to {all_samples_df.shape[0]} samples after excluding samples without valid sex information.")

sex_counts = all_samples_df['SEX'].value_counts()
print("\nSex distribution after filtering:")
print(sex_counts)



In [ ]:
merged_samples_df = pd.merge(
    all_samples_df,
    relationships_df,
    on='IID',
    how='left',
    suffixes=('_ped', '_rel')
)

samples_info_file = os.path.join(output_dir, 'all_samples_info.csv')
merged_samples_df.to_csv(samples_info_file, index=False)
print(f"Sample information saved to: {samples_info_file}")

In [ ]:
def improved_encoding(genotype_data):
    """Improved encoding for genetic data using additive inheritance model"""
    encoded_data = np.zeros((genotype_data.shape[0], genotype_data.shape[1] // 2), dtype=np.float32)
    
    for i in range(0, genotype_data.shape[1], 2):
        allele1 = genotype_data[:, i]
        allele2 = genotype_data[:, i+1]
        snp_idx = i // 2
        
        unique_alleles, counts = np.unique(np.concatenate([allele1, allele2]), return_counts=True)
        
        if len(unique_alleles) > 0:
            ref_allele = unique_alleles[np.argmax(counts)]
            
            alt_count = np.zeros(allele1.shape[0], dtype=np.float32)
            alt_count += (allele1 != ref_allele) & (allele1 != '0')
            alt_count += (allele2 != ref_allele) & (allele2 != '0')
            
            encoded_data[:, snp_idx] = alt_count
    
    return encoded_data

In [ ]:

valid_samples = all_samples_df['IID'].tolist()

all_genotype_data = []
all_samples_ids = []
all_sex_labels = []

for ped_file in tqdm(ped_files, desc="Processing PED files"):
    ped_columns = ['FID', 'IID', 'PAT', 'MAT', 'SEX', 'PHENO']
    ped_info = pd.read_csv(ped_file, sep='\s+', header=None, usecols=range(6), 
                          names=ped_columns, low_memory=False)
    
    valid_mask = ped_info['IID'].isin(valid_samples) & ped_info['SEX'].isin([1, 2])
    if valid_mask.sum() == 0:
        print(f"Skipping file {os.path.basename(ped_file)} - no valid samples")
        continue
    
    ped_info = ped_info[valid_mask]
    
    n_samples = ped_info.shape[0]
    sample_ids = ped_info['IID'].tolist()
    sex_values = ped_info['SEX'].tolist()
    
    all_samples_ids.extend(sample_ids)
    all_sex_labels.extend(sex_values)
    
    total_columns = 6 + (MAX_SNPS * 2)
    
    try:
        column_indices = list(range(6)) + list(range(6, 6 + (MAX_SNPS * 2)))
        genotype_data = pd.read_csv(ped_file, sep='\s+', header=None, usecols=column_indices, low_memory=False)
        
        genotype_data = genotype_data[valid_mask].reset_index(drop=True)
        
        snp_data = genotype_data.iloc[:, 6:].values
        
        encoded_data = improved_encoding(snp_data)
        
        all_genotype_data.append(encoded_data)
        
        print(f"Processed {encoded_data.shape[0]} samples and {encoded_data.shape[1]} SNPs from file {os.path.basename(ped_file)}")
        
        del genotype_data, snp_data
        gc.collect()
        
    except Exception as e:
        print(f"Error processing file {ped_file}: {str(e)}")
        continue


In [ ]:
all_genotypes = np.vstack(all_genotype_data)
print(f"Genetic data combined. Dimensions: {all_genotypes.shape}")

del all_genotype_data
gc.collect()

samples_df = pd.DataFrame({
    'IID': all_samples_ids,
    'SEX': all_sex_labels
})

samples_df['Population'] = samples_df['IID'].map(population_labels)

print("\nSex distribution in final data:")
print(samples_df['SEX'].value_counts())
print("\nSex distribution by population:")
print(pd.crosstab(samples_df['Population'], samples_df['SEX']))

In [ ]:
print("\nSelecting best SNPs for sex differentiation...")
X = all_genotypes
y = samples_df['SEX'].values

selector = SelectKBest(f_classif, k=10000)
X_selected = selector.fit_transform(X, y)

print(f"Selected {X_selected.shape[1]} SNPs out of {X.shape[1]} SNPs.")
print(f"Dimensions of selected data: {X_selected.shape}")

selected_indices = selector.get_support(indices=True)
selected_snps = map_df.iloc[selected_indices]
selected_snps.to_csv(os.path.join(output_dir, 'sex_selected_snps.csv'), index=False)
print(f"List of selected SNPs saved to: {os.path.join(output_dir, 'sex_selected_snps.csv')}")


In [ ]:
chr_counts = selected_snps['CHR'].value_counts().sort_index()
print(chr_counts)

plt.figure(figsize=(12, 6))
chr_counts.plot(kind='bar')
plt.title('Distribution of Selected SNPs Across Chromosomes')
plt.xlabel('Chromosome')
plt.ylabel('Number of SNPs')
plt.savefig(os.path.join(output_dir, 'sex_snps_distribution.png'))


In [ ]:
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X_selected)

print(f"Dimensions reduced to {X_pca.shape[1]} principal components.")
print(f"Explained variance ratio: {sum(pca.explained_variance_ratio_):.4f}")

plt.figure(figsize=(10, 6))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.title('PCA Cumulative Variance Curve')
plt.grid(True)
plt.savefig(os.path.join(output_dir, 'sex_pca_variance.png'))

In [ ]:
plt.figure(figsize=(12, 10))
sex_colors = {1: 'blue', 2: 'red'}  # 1=male, 2=female
sex_labels = {1: 'Male', 2: 'Female'}

for sex_value in [1, 2]:
    sex_mask = samples_df['SEX'] == sex_value
    plt.scatter(
        X_pca[sex_mask, 0], 
        X_pca[sex_mask, 1],
        c=sex_colors[sex_value],
        label=sex_labels[sex_value], 
        alpha=0.7
    )

plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('Sex Distribution in PCA Space')
plt.legend()
plt.savefig(os.path.join(output_dir, 'sex_pca_distribution.png'))


In [ ]:
features_df = pd.DataFrame(X_pca, columns=[f'PC_{i+1}' for i in range(X_pca.shape[1])])
features_df['IID'] = samples_df['IID'].values
features_df['SEX'] = samples_df['SEX'].values
features_df['Population'] = samples_df['Population'].values

features_file = os.path.join(output_dir, 'sex_features_pca.csv')
features_df.to_csv(features_file, index=False)
print(f"Extracted features saved to: {features_file}")

In [ ]:
feature_columns = [f'PC_{i+1}' for i in range(X_pca.shape[1])]
le_pop = LabelEncoder()
features_df['Population_encoded'] = le_pop.fit_transform(features_df['Population'])
feature_columns.append('Population_encoded')

X = features_df[feature_columns].values
y = features_df['SEX'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
if len(np.unique(y_train)) > 1 and abs(sum(y_train == 1) - sum(y_train == 2)) > 100:
    print("\nApplying SMOTE to handle data imbalance...")
    smote = SMOTE(random_state=42)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
    
    print(f"Size of training set after SMOTE: {X_train_balanced.shape[0]} samples")
    print("Sex distribution after SMOTE:")
    for sex_value, count in zip(*np.unique(y_train_balanced, return_counts=True)):
        print(f"{sex_labels.get(sex_value, sex_value)}: {count} samples")
else:
    X_train_balanced = X_train
    y_train_balanced = y_train
    print("No significant imbalance in the data, so SMOTE was not applied.")

In [ ]:
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 20],
    'model__min_samples_split': [2, 5],
    'model__class_weight': [None, 'balanced']
}

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state=42))
])

print("Searching for best parameters...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring='accuracy', n_jobs=-1
)
grid_search.fit(X_train_balanced, y_train_balanced)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation accuracy: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_

In [ ]:
y_pred = best_model.predict(X_test)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=['Male (1)', 'Female (2)']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Male', 'Female'],
            yticklabels=['Male', 'Female'])
plt.xlabel('Prediction')
plt.ylabel('True Value')
plt.title('Confusion Matrix for Sex Classification')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'sex_confusion_matrix.png'))

accuracy = accuracy_score(y_test, y_pred)
print(f"\nOverall model accuracy: {accuracy:.4f}")

In [ ]:
if hasattr(best_model, 'predict_proba'):
    y_prob = best_model.predict_proba(X_test)[:, 1]
    
    fpr, tpr, thresholds = roc_curve(y_test, y_prob, pos_label=2)
    auc = roc_auc_score(y_test, y_prob)
    
    plt.figure(figsize=(10, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve for Sex Classification')
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, 'sex_roc_curve.png'))

In [ ]:
if hasattr(best_model['model'], 'feature_importances_'):
    importances = best_model['model'].feature_importances_
    indices = np.argsort(importances)[::-1]
    
    print("\nTop 10 features for sex classification:")
    for i in range(min(10, len(feature_columns))):
        print(f"{feature_columns[indices[i]]}: {importances[indices[i]]:.4f}")
    
    plt.figure(figsize=(12, 6))
    plt.bar(range(min(20, len(feature_columns))), 
            importances[indices[:20]], 
            align='center')
    plt.xticks(range(min(20, len(feature_columns))), 
              [feature_columns[i] for i in indices[:20]], 
              rotation=90)
    plt.title('Feature Importance for Sex Classification')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'sex_feature_importance.png'))


In [ ]:
features_df['Predicted_SEX'] = best_model.predict(features_df[feature_columns].values)

for pop in features_df['Population'].unique():
    pop_mask = features_df['Population'] == pop
    pop_data = features_df[pop_mask]
    if not pop_data.empty:
        pop_accuracy = accuracy_score(pop_data['SEX'], pop_data['Predicted_SEX'])
        print(f"{pop}: accuracy = {pop_accuracy:.4f} ({sum(pop_mask)} samples)")

In [ ]:
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
svm = SVC(probability=True, class_weight='balanced', random_state=42)
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)

ensemble = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('svm', svm),
        ('mlp', mlp)
    ],
    voting='soft'
)

ensemble_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ensemble)
])

print("Training ensemble model...")
ensemble_pipeline.fit(X_train_balanced, y_train_balanced)

In [ ]:
ensemble_preds = ensemble_pipeline.predict(X_test)
ensemble_accuracy = accuracy_score(y_test, ensemble_preds)

print(f"Ensemble model accuracy: {ensemble_accuracy:.4f}")
print(classification_report(y_test, ensemble_preds, target_names=['Male (1)', 'Female (2)']))

cm_ensemble = confusion_matrix(y_test, ensemble_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_ensemble, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Male', 'Female'],
            yticklabels=['Male', 'Female'])
plt.xlabel('Prediction')
plt.ylabel('True Value')
plt.title('Confusion Matrix for Ensemble Model - Sex Classification')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'sex_ensemble_confusion_matrix.png'))


In [ ]:
import joblib
best_model_file = os.path.join(output_dir, 'best_sex_model.pkl')
ensemble_model_file = os.path.join(output_dir, 'ensemble_sex_model.pkl')

joblib.dump(best_model, best_model_file)
joblib.dump(ensemble_pipeline, ensemble_model_file)

print(f"\nBest model saved to: {best_model_file}")
print(f"Ensemble model saved to: {ensemble_model_file}")

In [ ]:
x_chr_snps = selected_snps[selected_snps['CHR'] == 'X']
y_chr_snps = selected_snps[selected_snps['CHR'] == 'Y']

print(f"Number of SNPs from X chromosome: {len(x_chr_snps)}")
print(f"Number of SNPs from Y chromosome: {len(y_chr_snps)}")

if len(x_chr_snps) > 0 or len(y_chr_snps) > 0:
    selected_snp_names = selected_snps['SNP'].tolist()
    
    if hasattr(best_model['model'], 'feature_importances_'):
        chr_x_indices = [i for i, snp in enumerate(selected_indices) if map_df.iloc[snp]['CHR'] == 'X']
        chr_y_indices = [i for i, snp in enumerate(selected_indices) if map_df.iloc[snp]['CHR'] == 'Y']
        
        if chr_x_indices:
            print("\nImportance of SNPs from X chromosome:")
            x_importances = [(i, importances[i]) for i in range(len(importances)) if i in chr_x_indices]
            x_importances.sort(key=lambda x: x[1], reverse=True)
            for i, imp in x_importances[:10]:
                snp_idx = selected_indices[i]
                snp_name = map_df.iloc[snp_idx]['SNP']
                print(f"{snp_name}: {imp:.4f}")
        
        if chr_y_indices:
            print("\nImportance of SNPs from Y chromosome:")
            y_importances = [(i, importances[i]) for i in range(len(importances)) if i in chr_y_indices]
            y_importances.sort(key=lambda x: x[1], reverse=True)
            for i, imp in y_importances[:10]:
                snp_idx = selected_indices[i]
                snp_name = map_df.iloc[snp_idx]['SNP']
                print(f"{snp_name}: {imp:.4f}")


In [ ]:
prediction_file = os.path.join(output_dir, 'sex_predictions.csv')

ensemble_preds_all = ensemble_pipeline.predict(X)
features_df['Predicted_SEX_Ensemble'] = ensemble_preds_all

best_preds_all = best_model.predict(X)
features_df['Predicted_SEX_Best'] = best_preds_all

features_df.to_csv(prediction_file, index=False)
print(f"\nData with predictions saved to: {prediction_file}")

In [ ]:
print("\n== Summary of Sex Classification Results from Genetic Data ==")
print(f"Total number of samples: {features_df.shape[0]}")
print(f"Sex distribution: Males = {sum(features_df['SEX'] == 1)}, Females = {sum(features_df['SEX'] == 2)}")
print(f"Random Forest model accuracy: {accuracy:.4f}")
print(f"Ensemble model accuracy: {ensemble_accuracy:.4f}")
print(f"Number of SNPs used: {X_selected.shape[1]} out of {X.shape[1]}")
print(f"Principal components used: {X_pca.shape[1]}")
print("\nSex classification model building and evaluation process completed.")

In [ ]:

data_dir = 'hapmap_data'
output_dir = os.path.join(data_dir, 'sex_prediction_data')
os.makedirs(output_dir, exist_ok=True)

In [ ]:
try:
    # First check if prediction file exists
    prediction_file = os.path.join(output_dir, 'sex_predictions.csv')
    if os.path.exists(prediction_file):
        features_df = pd.read_csv(prediction_file)
        print(f"Prediction file loaded: {prediction_file}")
        print(f"Data dimensions: {features_df.shape}")
        print(f"Column names: {features_df.columns.tolist()[:10]}...")
    else:
        # If prediction file doesn't exist, try reading the features file
        features_file = os.path.join(output_dir, 'sex_features_pca.csv')
        if os.path.exists(features_file):
            features_df = pd.read_csv(features_file)
            print(f"Features file loaded: {features_file}")
            print(f"Data dimensions: {features_df.shape}")
            print(f"Column names: {features_df.columns.tolist()[:10]}...")
        else:
            raise FileNotFoundError("Required data files not found")
except FileNotFoundError as e:
    print(f"Error: {str(e)}")
except Exception as e:
    print(f"Unexpected error: {str(e)}")

In [ ]:
try:
    plt.figure(figsize=(12, 10))
    
    # Define colors and labels
    sex_colors = {1: 'blue', 2: 'red'}
    sex_labels = {1: 'Male', 2: 'Female'}
    
    # Extract PCA columns
    pc_columns = [col for col in features_df.columns if col.startswith('PC_')]
    
    if len(pc_columns) >= 2:
        # Plot first two principal components
        for sex_value in [1, 2]:
            sex_mask = features_df['SEX'] == sex_value
            plt.scatter(
                features_df.loc[sex_mask, pc_columns[0]], 
                features_df.loc[sex_mask, pc_columns[1]],
                c=sex_colors[sex_value],
                label=sex_labels[sex_value], 
                alpha=0.7
            )
        
        plt.xlabel('First Principal Component')
        plt.ylabel('Second Principal Component')
        plt.title('Sex Distribution in PCA Space')
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'sex_pca_english.png'), dpi=300)
        print(f"PCA plot saved to: {os.path.join(output_dir, 'sex_pca_english.png')}")
except Exception as e:
    print(f"Error in PCA visualization: {str(e)}")


In [ ]:
try:
    # Read selected SNPs file
    selected_snps_file = os.path.join(output_dir, 'sex_selected_snps.csv')
    if os.path.exists(selected_snps_file):
        selected_snps = pd.read_csv(selected_snps_file)
        
        # Calculate SNP distribution across chromosomes
        chr_counts = selected_snps['CHR'].value_counts().sort_index()
        
        plt.figure(figsize=(14, 7))
        ax = chr_counts.plot(kind='bar', color='skyblue')
        plt.title('Distribution of Selected SNPs Across Chromosomes')
        plt.xlabel('Chromosome')
        plt.ylabel('Number of SNPs')
        plt.xticks(rotation=45)
        
        # Add values above each bar
        for i, v in enumerate(chr_counts):
            ax.text(i, v + 0.1, str(v), ha='center')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'sex_snps_distribution_english.png'), dpi=300)
        print(f"SNP distribution plot saved to: {os.path.join(output_dir, 'sex_snps_distribution_english.png')}")
except Exception as e:
    print(f"Error in SNP distribution visualization: {str(e)}")

In [ ]:
try:
    if 'Population' in features_df.columns:
        pop_counts = features_df['Population'].value_counts()
        
        plt.figure(figsize=(12, 6))
        ax = pop_counts.plot(kind='bar', color='lightgreen')
        plt.title('Population Distribution')
        plt.xlabel('Population')
        plt.ylabel('Number of Samples')
        plt.xticks(rotation=45)
        
        # Add values above each bar
        for i, v in enumerate(pop_counts):
            ax.text(i, v + 0.1, str(v), ha='center')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'population_distribution_english.png'), dpi=300)
        print(f"Population distribution plot saved to: {os.path.join(output_dir, 'population_distribution_english.png')}")
except Exception as e:
    print(f"Error in population distribution visualization: {str(e)}")

In [ ]:
    if 'Population' in features_df.columns:
        sex_pop_counts = pd.crosstab(features_df['Population'], features_df['SEX'])
        sex_pop_counts.columns = [sex_labels[col] for col in sex_pop_counts.columns]
        
        plt.figure(figsize=(14, 7))
        sex_pop_counts.plot(kind='bar', stacked=True)
        plt.title('Sex Distribution by Population')
        plt.xlabel('Population')
        plt.ylabel('Number of Samples')
        plt.xticks(rotation=45)
        plt.legend(title='Sex')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'sex_by_population_english.png'), dpi=300)
        print(f"Sex by population plot saved to: {os.path.join(output_dir, 'sex_by_population_english.png')}")


In [ ]:
 if 'Predicted_SEX' in features_df.columns or 'Predicted_SEX_Ensemble' in features_df.columns:
        pred_col = 'Predicted_SEX_Ensemble' if 'Predicted_SEX_Ensemble' in features_df.columns else 'Predicted_SEX'
        
        plt.figure(figsize=(14, 7))
        
        # Calculate accuracy for each population
        population_accuracy = {}
        
        for pop in features_df['Population'].unique():
            pop_mask = features_df['Population'] == pop
            pop_data = features_df[pop_mask]
            
            if not pop_data.empty and len(pop_data) > 5:  # Avoid very small populations
                correct = (pop_data['SEX'] == pop_data[pred_col]).sum()
                total = len(pop_data)
                accuracy = correct / total
                population_accuracy[pop] = accuracy
        
        # Plot accuracy percentages
        pops = list(population_accuracy.keys())
        accuracies = list(population_accuracy.values())
        
        plt.figure(figsize=(14, 7))
        plt.bar(pops, accuracies, color='lightcoral')
        plt.title('Sex Classification Accuracy by Population')
        plt.xlabel('Population')
        plt.ylabel('Accuracy')
        plt.xticks(rotation=45)
        plt.ylim([0, 1.05])
        
        # Add values above each bar
        for i, v in enumerate(accuracies):
            plt.text(i, v + 0.02, f'{v:.2f}', ha='center')
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'accuracy_by_population_english.png'), dpi=300)
        print(f"Accuracy by population plot saved to: {os.path.join(output_dir, 'accuracy_by_population_english.png')}")

In [ ]:
if len(pc_columns) >= 3:
        fig = plt.figure(figsize=(12, 10))
        ax = fig.add_subplot(111, projection='3d')
        
        for sex_value in [1, 2]:
            sex_mask = features_df['SEX'] == sex_value
            ax.scatter(
                features_df.loc[sex_mask, pc_columns[0]],
                features_df.loc[sex_mask, pc_columns[1]],
                features_df.loc[sex_mask, pc_columns[2]],
                c=sex_colors[sex_value],
                label=sex_labels[sex_value],
                alpha=0.7
            )
        
        ax.set_xlabel(pc_columns[0])
        ax.set_ylabel(pc_columns[1])
        ax.set_zlabel(pc_columns[2])
        ax.set_title('Sex Distribution in 3D PCA Space')
        ax.legend()
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'sex_pca_3d_english.png'), dpi=300)
        print(f"3D PCA plot saved to: {os.path.join(output_dir, 'sex_pca_3d_english.png')}")


In [ ]:
top_pops = features_df['Population'].value_counts().head(8).index.tolist()

fig = plt.figure(figsize=(15, 12))

for i, pop in enumerate(top_pops):
    ax = fig.add_subplot(3, 3, i+1)
    
    pop_data = features_df[features_df['Population'] == pop]
    
    for sex_value in [1, 2]:
        sex_mask = pop_data['SEX'] == sex_value
        sex_label = 'Male' if sex_value == 1 else 'Female'
        sex_color = 'blue' if sex_value == 1 else 'red'
        
        ax.scatter(
            pop_data.loc[sex_mask, pc_columns[0]],
            pop_data.loc[sex_mask, pc_columns[1]],
            c=sex_color,
            label=sex_label,
            alpha=0.7,
            s=30
        )
    
    # Add sample count to title
    ax.set_title(f'Population {pop} (n={len(pop_data)})')
    ax.set_xlabel(pc_columns[0])
    ax.set_ylabel(pc_columns[1])
    ax.legend()

# Main title
plt.suptitle('PCA by Population and Sex', fontsize=16, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(output_dir, 'pca_by_population_sex_english.png'), dpi=300)
print(f"PCA by population and sex plot saved to: {os.path.join(output_dir, 'pca_by_population_sex_english.png')}")

In [ ]:
if len(pc_columns) >= 4:
        plt.figure(figsize=(14, 14))
        
        # Create new DataFrame with components and sex
        plot_df = features_df[pc_columns[:4] + ['SEX']].copy()
        plot_df['SEX'] = plot_df['SEX'].map({1: 'Male', 2: 'Female'})
        
        # Plot scatter matrix
        sns.pairplot(plot_df, hue='SEX', palette={'Male': 'blue', 'Female': 'red'}, 
                    diag_kind='kde', plot_kws={'alpha': 0.6}, height=2.5)
        
        plt.suptitle('Pairwise Scatter Plot of First Four Principal Components by Sex', y=1.02, fontsize=16)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'sex_pca_pairplot_english.png'), dpi=300)
        print(f"PCA pairplot saved to: {os.path.join(output_dir, 'sex_pca_pairplot_english.png')}")


In [ ]:
prediction_file = os.path.join(output_dir, 'sex_predictions.csv')
features_file = os.path.join(output_dir, 'sex_features_pca.csv')

if os.path.exists(prediction_file):
    df = pd.read_csv(prediction_file)
    print(f"Prediction file loaded: {prediction_file}")
elif os.path.exists(features_file):
    df = pd.read_csv(features_file)
    print(f"Features file loaded: {features_file}")
else:
    raise FileNotFoundError("Required data files not found")

In [ ]:
pred_col = 'Predicted_SEX_Ensemble' if 'Predicted_SEX_Ensemble' in df.columns else 'Predicted_SEX_Best'

# Create column to indicate errors
df['Error'] = (df['SEX'] != df[pred_col]).astype(int)

# Aggregate errors by population
error_by_pop = df.groupby('Population')['Error'].mean()

plt.figure(figsize=(14, 7))
error_by_pop.sort_values(ascending=False).plot(kind='bar', color='salmon')
plt.title('Error Rate in Sex Classification by Population')
plt.xlabel('Population')
plt.ylabel('Error Rate')
plt.xticks(rotation=45)
plt.ylim([0, min(1.0, error_by_pop.max() * 1.2)])  # Set y-axis limits

# Add values above each bar
for i, v in enumerate(error_by_pop.sort_values(ascending=False)):
    plt.text(i, v + 0.01, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'error_by_population_english.png'), dpi=300)
print(f"Error distribution plot saved to: {os.path.join(output_dir, 'error_by_population_english.png')}")

In [ ]:
pred_col = 'Predicted_SEX_Ensemble' if 'Predicted_SEX_Ensemble' in df.columns else 'Predicted_SEX_Best'

# Create confusion matrix
cm = confusion_matrix(df['SEX'], df[pred_col])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=['Male', 'Female'],
           yticklabels=['Male', 'Female'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix for Sex Classification')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'confusion_matrix_english.png'), dpi=300)
print(f"Confusion matrix saved to: {os.path.join(output_dir, 'confusion_matrix_english.png')}")


In [ ]:
try:
    pc_columns = [col for col in df.columns if col.startswith('PC_')]
    
    if len(pc_columns) >= 3 and 'SEX' in df.columns:
        # Create a dashboard style visualization
        fig = plt.figure(figsize=(18, 12))
        gs = GridSpec(2, 3, figure=fig)
        
        # Define colors and markers
        sex_colors = {1: 'blue', 2: 'red'}
        sex_labels = {1: 'Male', 2: 'Female'}
        
        # Plot 1: PCA scatter plot (PC1 vs PC2)
        ax1 = fig.add_subplot(gs[0, 0])
        for sex_value in [1, 2]:
            sex_mask = df['SEX'] == sex_value
            ax1.scatter(
                df.loc[sex_mask, pc_columns[0]],
                df.loc[sex_mask, pc_columns[1]],
                c=sex_colors[sex_value],
                label=sex_labels[sex_value],
                alpha=0.7,
                s=20
            )
        ax1.set_xlabel(pc_columns[0])
        ax1.set_ylabel(pc_columns[1])
        ax1.set_title('PC1 vs PC2')
        ax1.legend()
        
        # Plot 2: PCA scatter plot (PC1 vs PC3)
        ax2 = fig.add_subplot(gs[0, 1])
        for sex_value in [1, 2]:
            sex_mask = df['SEX'] == sex_value
            ax2.scatter(
                df.loc[sex_mask, pc_columns[0]],
                df.loc[sex_mask, pc_columns[2]],
                c=sex_colors[sex_value],
                label=sex_labels[sex_value],
                alpha=0.7,
                s=20
            )
        ax2.set_xlabel(pc_columns[0])
        ax2.set_ylabel(pc_columns[2])
        ax2.set_title('PC1 vs PC3')
        ax2.legend()
        
        # Plot 3: PCA 3D scatter plot
        ax3 = fig.add_subplot(gs[0, 2], projection='3d')
        for sex_value in [1, 2]:
            sex_mask = df['SEX'] == sex_value
            ax3.scatter(
                df.loc[sex_mask, pc_columns[0]],
                df.loc[sex_mask, pc_columns[1]],
                df.loc[sex_mask, pc_columns[2]],
                c=sex_colors[sex_value],
                label=sex_labels[sex_value],
                alpha=0.7,
                s=20
            )
        ax3.set_xlabel(pc_columns[0], fontsize=8)
        ax3.set_ylabel(pc_columns[1], fontsize=8)
        ax3.set_zlabel(pc_columns[2], fontsize=8)
        ax3.set_title('3D PCA')
        ax3.legend()
        
        # Plot 4: Box plots for first 3 PCs by sex
        ax4 = fig.add_subplot(gs[1, 0])
        box_data = []
        labels = []
        
        for i, pc in enumerate(pc_columns[:3]):
            for sex_value in [1, 2]:
                box_data.append(df[df['SEX'] == sex_value][pc])
                labels.append(f"{sex_labels[sex_value]}\n{pc}")
        
        ax4.boxplot(box_data, labels=labels)
        ax4.set_title('PC Values by Sex')
        ax4.set_ylabel('Value')
        plt.setp(ax4.get_xticklabels(), rotation=45, fontsize=8)
        
        # Plot 5: Variance explained by PCs
        ax5 = fig.add_subplot(gs[1, 1])
        # Since we don't have direct access to PCA object, we'll create a mock variance plot
        dummy_variance = np.array([0.25, 0.15, 0.10, 0.08, 0.05, 0.04, 0.03, 0.02, 0.02, 0.01])[:len(pc_columns)]
        dummy_variance = dummy_variance / dummy_variance.sum()  # normalize to sum to 1
        
        ax5.bar(range(1, len(dummy_variance) + 1), dummy_variance, color='skyblue')
        ax5.plot(range(1, len(dummy_variance) + 1), np.cumsum(dummy_variance), 'ro-', linewidth=2)
        ax5.set_xlabel('Principal Component')
        ax5.set_ylabel('Proportion of Variance Explained')
        ax5.set_title('PCA Variance Explained')
        ax5.set_xticks(range(1, len(dummy_variance) + 1))
        ax5.set_ylim([0, 1.05])
        ax5.grid(True, alpha=0.3)
        
        # Plot 6: PC distribution density
        ax6 = fig.add_subplot(gs[1, 2])
        for sex_value in [1, 2]:
            sex_mask = df['SEX'] == sex_value
            for i, pc in enumerate(pc_columns[:3]):
                sns.kdeplot(
                    df.loc[sex_mask, pc],
                    ax=ax6,
                    label=f"{sex_labels[sex_value]} - {pc}",
                    color=f"{sex_colors[sex_value]}" if i==0 else (f"lightblue" if sex_value==1 else "lightcoral")
                )
        ax6.set_xlabel('Value')
        ax6.set_ylabel('Density')
        ax6.set_title('PC Distribution by Sex')
        ax6.legend(fontsize=8)
        
        plt.suptitle('PCA Analysis Dashboard for Sex Classification', fontsize=16, y=0.98)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(os.path.join(output_dir, 'pca_dashboard_english.png'), dpi=300)
        print(f"PCA dashboard saved to: {os.path.join(output_dir, 'pca_dashboard_english.png')}")
except Exception as e:
    print(f"Error in PCA dashboard visualization: {str(e)}")

In [ ]:

pc_columns = [col for col in df.columns if col.startswith('PC_')]

if len(pc_columns) >= 10:
    # Extract PC data for t-SNE
    X_pca = df[pc_columns].values
    
    # Apply t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    X_tsne = tsne.fit_transform(X_pca)
    
    # Create DataFrame for easy plotting
    tsne_df = pd.DataFrame({
        'TSNE_1': X_tsne[:, 0],
        'TSNE_2': X_tsne[:, 1],
        'SEX': df['SEX']
    })
    
    if 'Population' in df.columns:
        tsne_df['Population'] = df['Population']
    
    # Plot t-SNE results by sex
    plt.figure(figsize=(10, 8))
    
    for sex_value in [1, 2]:
        sex_mask = tsne_df['SEX'] == sex_value
        sex_label = 'Male' if sex_value == 1 else 'Female'
        sex_color = 'blue' if sex_value == 1 else 'red'
        
        plt.scatter(
            tsne_df.loc[sex_mask, 'TSNE_1'],
            tsne_df.loc[sex_mask, 'TSNE_2'],
            c=sex_color,
            label=sex_label,
            alpha=0.7,
            s=30
        )
    
    plt.xlabel('t-SNE Component 1')
    plt.ylabel('t-SNE Component 2')
    plt.title('t-SNE Visualization of Sex Classification')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'tsne_sex_english.png'), dpi=300)
    print(f"t-SNE visualization saved to: {os.path.join(output_dir, 'tsne_sex_english.png')}")
    
    # If population data is available, plot t-SNE by population
    if 'Population' in tsne_df.columns:
        # Get top populations for clear visualization
        top_pops = df['Population'].value_counts().head(6).index
        
        plt.figure(figsize=(12, 10))
        
        # Create a color map for populations
        pop_cmap = plt.cm.get_cmap('tab10', len(top_pops))
        
        for i, pop in enumerate(top_pops):
            pop_mask = tsne_df['Population'] == pop
            
            plt.scatter(
                tsne_df.loc[pop_mask, 'TSNE_1'],
                tsne_df.loc[pop_mask, 'TSNE_2'],
                c=[pop_cmap(i)],
                label=pop,
                alpha=0.7,
                s=30
            )
        
        plt.xlabel('t-SNE Component 1')
        plt.ylabel('t-SNE Component 2')
        plt.title('t-SNE Visualization by Population')
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'tsne_population_english.png'), dpi=300)
        print(f"t-SNE by population visualization saved to: {os.path.join(output_dir, 'tsne_population_english.png')}")

In [ ]:

pc_columns = [col for col in df.columns if col.startswith('PC_')]

if len(pc_columns) >= 3:
    # Extract PC data for clustering
    X_cluster = df[pc_columns[:3]].values
    
    # Apply K-means clustering (k=2 for sex classification)
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_cluster)
    
    # Create 3D plot
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot clusters
    scatter = ax.scatter(
        X_cluster[:, 0],
        X_cluster[:, 1],
        X_cluster[:, 2],
        c=cluster_labels,
        cmap='coolwarm',
        alpha=0.7,
        s=30
    )
    
    # Plot cluster centers
    ax.scatter(
        kmeans.cluster_centers_[:, 0],
        kmeans.cluster_centers_[:, 1],
        kmeans.cluster_centers_[:, 2],
        c='black',
        marker='X',
        s=100,
        label='Cluster Centers'
    )
    
    ax.set_xlabel(pc_columns[0])
    ax.set_ylabel(pc_columns[1])
    ax.set_zlabel(pc_columns[2])
    ax.set_title('K-means Clustering (k=2) on PCA Components')
    
    # Create legend
    legend1 = ax.legend(*scatter.legend_elements(), title="Clusters")
    ax.add_artist(legend1)
    ax.legend([plt.Line2D([0], [0], linestyle="none", c='black', marker='X')], 
             ['Cluster Centers'], numpoints=1)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'kmeans_clustering_english.png'), dpi=300)
    print(f"K-means clustering visualization saved to: {os.path.join(output_dir, 'kmeans_clustering_english.png')}")


In [ ]:

pc_columns = [col for col in df.columns if col.startswith('PC_')]

if len(pc_columns) >= 3 and 'SEX' in df.columns:
    # Extract PC data for clustering
    X_cluster = df[pc_columns[:3]].values
    
    # Apply K-means clustering (k=2 for sex classification)
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_cluster)
    
    # Create a confusion matrix-like comparison
    true_sex = df['SEX'].values
    
    # We need to align clusters with sex labels (cluster 0/1 might not correspond to male/female)
    # Count how many of each sex are in each cluster
    cluster_0_male = np.sum((cluster_labels == 0) & (true_sex == 1))
    cluster_0_female = np.sum((cluster_labels == 0) & (true_sex == 2))
    cluster_1_male = np.sum((cluster_labels == 1) & (true_sex == 1))
    cluster_1_female = np.sum((cluster_labels == 1) & (true_sex == 2))
    
    # Determine which cluster corresponds to which sex
    if (cluster_0_male + cluster_1_female) > (cluster_0_female + cluster_1_male):
        # Cluster 0 is mostly male, Cluster 1 is mostly female
        cluster_male = 0
        cluster_female = 1
    else:
        # Cluster 0 is mostly female, Cluster 1 is mostly male
        cluster_male = 1
        cluster_female = 0
    
    # Create a new array with predicted sex (1=male, 2=female)
    cluster_sex = np.where(cluster_labels == cluster_male, 1, 2)
    
    # Calculate accuracy
    accuracy = np.mean(cluster_sex == true_sex)
    
    # Create confusion matrix
    cm = confusion_matrix(true_sex, cluster_sex)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=['Male', 'Female'],
               yticklabels=['Male', 'Female'])
    plt.xlabel('Predicted (K-means)')
    plt.ylabel('True Sex')
    plt.title(f'K-means Clustering vs. True Sex Labels\nAccuracy: {accuracy:.4f}')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'kmeans_vs_true_sex_english.png'), dpi=300)
    print(f"K-means vs. true sex comparison saved to: {os.path.join(output_dir, 'kmeans_vs_true_sex_english.png')}")



In [ ]:
feature_importance_file = os.path.join(output_dir, 'sex_feature_importance.png')
if os.path.exists(feature_importance_file):
    print(f"Feature importance plot already exists: {feature_importance_file}")
else:
    # If we have SNP selection data, we can create a basic feature importance plot
    selected_snps_file = os.path.join(output_dir, 'sex_selected_snps.csv')
    if os.path.exists(selected_snps_file):
        snps_df = pd.read_csv(selected_snps_file)
        
        # Generate mock feature importance scores if actual ones aren't available
        # In a real scenario, these would come from the model
        top_n = 20
        if len(snps_df) >= top_n:
            plt.figure(figsize=(12, 8))
            
            # Use chromosome as a factor in mock importance
            chr_weights = {'X': 1.0, 'Y': 0.9}
            mock_importance = np.random.exponential(scale=0.5, size=len(snps_df))
            
            # Adjust importance for sex chromosomes
            for i, chr_val in enumerate(snps_df['CHR']):
                if str(chr_val) in chr_weights:
                    mock_importance[i] *= chr_weights[str(chr_val)] * 3
            
            # Sort and select top features
            sorted_indices = np.argsort(mock_importance)[::-1][:top_n]
            top_snps = snps_df.iloc[sorted_indices]
            top_importances = mock_importance[sorted_indices]
            
            # Create feature labels
            feature_labels = [f"{snp} (Chr{chr})" for snp, chr in zip(top_snps['SNP'], top_snps['CHR'])]
            
            # Plot
            colors = ['skyblue' if str(chr) not in ['X', 'Y'] else 'salmon' for chr in top_snps['CHR']]
            plt.barh(range(len(top_importances)), top_importances, color=colors)
            plt.yticks(range(len(top_importances)), feature_labels)
            plt.xlabel('Feature Importance (Mock Data)')
            plt.ylabel('SNP')
            plt.title('Top 20 SNPs by Importance for Sex Classification')
            
            # Add a note that this is simulated data
            plt.figtext(0.5, 0.01, 'Note: Feature importance values are simulated for illustration purposes', 
                       ha='center', fontsize=10, style='italic')
            
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, 'feature_importance_simulation_english.png'), dpi=300)
            print(f"Simulated feature importance visualization saved to: {os.path.join(output_dir, 'feature_importance_simulation_english.png')}")



In [ ]:
# حفظ SelectKBest
selector_file = os.path.join(output_dir, 'feature_selector.pkl')
joblib.dump(selector, selector_file)

# حفظ PCA
pca_file = os.path.join(output_dir, 'pca_model.pkl')
joblib.dump(pca, pca_file)

In [ ]:
import os
import zipfile
import datetime

# تحديد مجلدات المشروع
data_dir = 'hapmap_data'
output_dir = os.path.join(data_dir, 'sex_prediction_data')

# إنشاء اسم للملف المضغوط يتضمن التاريخ والوقت الحالي
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"dna_age_prediction_results_{timestamp}.zip"

# إنشاء قائمة بالملفات المراد تضمينها
files_to_include = []

# 1. البحث عن ملفات النماذج (.pkl)
model_files = [f for f in os.listdir(output_dir) if f.endswith('.pkl')]
for file in model_files:
    files_to_include.append(os.path.join(output_dir, file))

# 2. البحث عن ملفات CSV
csv_files = [f for f in os.listdir(output_dir) if f.endswith('.csv')]
for file in csv_files:
    files_to_include.append(os.path.join(output_dir, file))

# 3. البحث عن ملفات الصور
image_files = [f for f in os.listdir(output_dir) if f.endswith(('.png', '.jpg', '.jpeg'))]
for file in image_files:
    files_to_include.append(os.path.join(output_dir, file))

# إنشاء الملف المضغوط
print(f"إنشاء ملف مضغوط: {zip_filename}")
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # إضافة الملفات إلى الملف المضغوط
    for file in files_to_include:
        if os.path.exists(file):
            # احفظ الملف في الملف المضغوط باستخدام اسم الملف فقط (بدون المسار)
            zipf.write(file, os.path.basename(file))
            print(f"تمت إضافة: {os.path.basename(file)}")
        else:
            print(f"تحذير: الملف غير موجود: {file}")

# طباعة ملخص
print(f"\nاكتمل إنشاء الملف المضغوط: {zip_filename}")
print(f"عدد الملفات المضمنة: {len(files_to_include)}")
print(f"أنواع الملفات: نماذج تعلم الآلة ({len(model_files)}), ملفات CSV ({len(csv_files)}), صور ({len(image_files)})")

In [ ]:
import os
import zipfile
import shutil
import pandas as pd
import joblib
from datetime import datetime

# تحديد المسارات
data_dir = 'hapmap_data'
output_dir = os.path.join(data_dir, 'sex_prediction_data')
os.makedirs(output_dir, exist_ok=True)

# إنشاء مجلد مؤقت للملفات التي سيتم تضمينها في ملف ZIP
temp_dir = os.path.join(output_dir, 'package_temp')
os.makedirs(temp_dir, exist_ok=True)

# إنشاء مجلدات منظمة داخل المجلد المؤقت
models_dir = os.path.join(temp_dir, 'models')
os.makedirs(models_dir, exist_ok=True)

images_dir = os.path.join(temp_dir, 'visualizations')
os.makedirs(images_dir, exist_ok=True)

data_files_dir = os.path.join(temp_dir, 'data')
os.makedirs(data_files_dir, exist_ok=True)

# نسخ ملفات النماذج
model_files = [
    'best_sex_model.pkl',
    'ensemble_sex_model.pkl',
    'feature_selector.pkl',  # إذا كان موجودًا
    'pca_model.pkl'          # إذا كان موجودًا
]

for model_file in model_files:
    src_path = os.path.join(output_dir, model_file)
    if os.path.exists(src_path):
        shutil.copy2(src_path, os.path.join(models_dir, model_file))
        print(f"تم نسخ {model_file} إلى مجلد النماذج")

# نسخ ملفات البيانات
data_files = [
    'sex_selected_snps.csv',
    'sex_predictions.csv',
    'sex_features_pca.csv',
    'all_samples_info.csv'
]

for data_file in data_files:
    src_path = os.path.join(output_dir, data_file)
    if os.path.exists(src_path):
        shutil.copy2(src_path, os.path.join(data_files_dir, data_file))
        print(f"تم نسخ {data_file} إلى مجلد البيانات")

# نسخ الصور
image_extensions = ['.png', '.jpg', '.jpeg']
image_files = []

for file in os.listdir(output_dir):
    if any(file.endswith(ext) for ext in image_extensions):
        src_path = os.path.join(output_dir, file)
        shutil.copy2(src_path, os.path.join(images_dir, file))
        image_files.append(file)
        print(f"تم نسخ {file} إلى مجلد الصور")

# إنشاء ملف README.md
readme_content = f"""# مشروع التنبؤ بالجنس من بيانات SNP

تاريخ الإنشاء: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## محتويات الحزمة

### نماذج التعلم الآلي:
{os.linesep.join(['- ' + file for file in model_files if os.path.exists(os.path.join(models_dir, file))])}

### ملفات البيانات:
{os.linesep.join(['- ' + file for file in data_files if os.path.exists(os.path.join(data_files_dir, file))])}

### المخططات والرسوم البيانية:
{os.linesep.join(['- ' + file for file in image_files])}

## كيفية الاستخدام

لاستخدام هذه النماذج للتنبؤ بجنس شخص من بيانات SNP الخاصة به، يمكنك استخدام الكود التالي:

```python
import joblib
import pandas as pd
import numpy as np

# تحميل النماذج
best_model = joblib.load('models/best_sex_model.pkl')
ensemble_model = joblib.load('models/ensemble_sex_model.pkl')
feature_selector = joblib.load('models/feature_selector.pkl')  # إذا كان متاحًا
pca_model = joblib.load('models/pca_model.pkl')  # إذا كان متاحًا

# تحميل قائمة SNPs المختارة
selected_snps = pd.read_csv('data/sex_selected_snps.csv')

# بعد معالجة بيانات SNP للمستخدم وتحويلها إلى مصفوفة X:
# X_selected = feature_selector.transform(X)
# X_pca = pca_model.transform(X_selected)

# التنبؤ باستخدام النموذج الأفضل
# prediction = best_model.predict(X_pca_with_population)
# sex_label = "ذكر" if prediction[0] == 1 else "أنثى"
```

راجع الملفات في مجلد البيانات للحصول على معلومات إضافية حول التدريب والتقييم.
"""

with open(os.path.join(temp_dir, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(readme_content)
print("تم إنشاء ملف README.md")

# إضافة كود تنبؤ نموذجي
prediction_code = """import os
import joblib
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

def predict_sex_from_genetic_data(genetic_data_file, output_file=None):
    \"\"\"
    التنبؤ بالجنس من ملف بيانات وراثية
    
    المعلمات:
        genetic_data_file (str): مسار ملف البيانات الوراثية بتنسيق متوافق
        output_file (str): مسار ملف الإخراج (اختياري)
    
    يعيد:
        DataFrame: نتائج التنبؤ مع معرفات العينة والجنس المتوقع
    \"\"\"
    # تحميل النماذج
    models_dir = 'models'
    data_dir = 'data'
    
    # تحميل النموذج الرئيسي
    ensemble_model_path = os.path.join(models_dir, 'ensemble_sex_model.pkl')
    if os.path.exists(ensemble_model_path):
        model = joblib.load(ensemble_model_path)
    else:
        model = joblib.load(os.path.join(models_dir, 'best_sex_model.pkl'))
    
    # تحميل محول المميزات (إذا كان متاحًا)
    feature_selector_path = os.path.join(models_dir, 'feature_selector.pkl')
    pca_model_path = os.path.join(models_dir, 'pca_model.pkl')
    
    has_feature_selector = os.path.exists(feature_selector_path)
    has_pca = os.path.exists(pca_model_path)
    
    if has_feature_selector:
        feature_selector = joblib.load(feature_selector_path)
    
    if has_pca:
        pca_model = joblib.load(pca_model_path)
    
    # تحميل قائمة SNPs المختارة
    selected_snps = pd.read_csv(os.path.join(data_dir, 'sex_selected_snps.csv'))
    
    # قراءة بيانات المدخلات (هذا مثال، يجب تعديله حسب تنسيق البيانات الحقيقي)
    # افتراضياً نفترض أن البيانات قد تمت معالجتها مسبقًا
    input_data = pd.read_csv(genetic_data_file)
    
    # استخراج المميزات والمعرفات
    sample_ids = input_data['IID'].values if 'IID' in input_data.columns else np.arange(len(input_data))
    
    # استخراج بيانات SNP (يجب تعديل هذا حسب تنسيق البيانات)
    if 'PC_1' in input_data.columns:
        # البيانات محولة مسبقًا إلى مكونات رئيسية
        X = input_data[[col for col in input_data.columns if col.startswith('PC_')]].values
    else:
        # نفترض أن البيانات تحتاج إلى معالجة
        # هذا مجرد مثال، يجب تعديله حسب تنسيق البيانات الحقيقي
        snp_columns = selected_snps['SNP'].tolist()
        if all(snp in input_data.columns for snp in snp_columns):
            X = input_data[snp_columns].values
        else:
            raise ValueError("تنسيق البيانات غير متوافق مع النموذج")
        
        # تطبيق محول المميزات إذا كان متاحًا
        if has_feature_selector:
            X = feature_selector.transform(X)
        
        # تطبيق PCA إذا كان متاحًا
        if has_pca:
            X = pca_model.transform(X)
    
    # إضافة معلومات السكان إذا كانت متاحة
    if 'Population' in input_data.columns:
        le_pop = LabelEncoder()
        population_encoded = le_pop.fit_transform(input_data['Population']).reshape(-1, 1)
        X = np.hstack([X, population_encoded])
    
    # التنبؤ
    y_pred = model.predict(X)
    
    # إعداد النتائج
    results = pd.DataFrame({
        'IID': sample_ids,
        'Predicted_SEX': y_pred,
        'Predicted_SEX_Label': ['Male' if sex == 1 else 'Female' for sex in y_pred]
    })
    
    # حفظ النتائج إذا تم تحديد ملف الإخراج
    if output_file:
        results.to_csv(output_file, index=False)
        print(f"تم حفظ نتائج التنبؤ إلى: {output_file}")
    
    return results

# مثال للاستخدام
if __name__ == "__main__":
    # مثال لكيفية استخدام الدالة (يجب تعديل المسارات)
    # genetic_data_file = "path/to/input_data.csv"
    # output_file = "path/to/sex_predictions_output.csv"
    # predictions = predict_sex_from_genetic_data(genetic_data_file, output_file)
    # print(predictions.head())
    
    print("برنامج التنبؤ بالجنس من SNP")
    print("يرجى تعديل الكود لتحديد مسارات الملفات المناسبة قبل الاستخدام")
"""

with open(os.path.join(temp_dir, 'predict_sex.py'), 'w', encoding='utf-8') as f:
    f.write(prediction_code)
print("تم إنشاء ملف كود التنبؤ")

# إنشاء ملف ZIP
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_filename = os.path.join(output_dir, f'sex_prediction_package_{timestamp}.zip')

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(temp_dir):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, temp_dir)
            zipf.write(file_path, arcname)

print(f"\nتم إنشاء ملف ZIP بنجاح: {zip_filename}")

# تنظيف المجلد المؤقت
shutil.rmtree(temp_dir)
print("تم حذف المجلد المؤقت")

print("\nاكتمل إنشاء حزمة المشروع!")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import gc
import glob
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

def improved_encoding(genotype_data):
    """Improved encoding for genetic data using additive inheritance model"""
    encoded_data = np.zeros((genotype_data.shape[0], genotype_data.shape[1] // 2), dtype=np.float32)
    
    for i in range(0, genotype_data.shape[1], 2):
        allele1 = genotype_data[:, i]
        allele2 = genotype_data[:, i+1]
        snp_idx = i // 2
        
        unique_alleles, counts = np.unique(np.concatenate([allele1, allele2]), return_counts=True)
        
        if len(unique_alleles) > 0:
            ref_allele = unique_alleles[np.argmax(counts)]
            
            alt_count = np.zeros(allele1.shape[0], dtype=np.float32)
            alt_count += (allele1 != ref_allele) & (allele1 != '0')
            alt_count += (allele2 != ref_allele) & (allele2 != '0')
            
            encoded_data[:, snp_idx] = alt_count
    
    return encoded_data

def predict_sex_from_dna(input_file, output_dir=None, max_snps=300000, visualize=True, csv_input=False):
    """
    Predict sex from DNA SNP data
    
    Args:
        input_file (str): Path to the PED file or CSV file
        output_dir (str): Directory where models are stored and outputs will be saved
        max_snps (int): Maximum number of SNPs to use
        visualize (bool): Create and show visualization plots
        csv_input (bool): Set to True if input is a CSV file instead of PED
    
    Returns:
        DataFrame: Results with predictions and evaluation metrics
    """
    # Set up paths based on Kaggle environment
    if os.path.exists('/kaggle'):
        # We're in Kaggle
        if output_dir is None:
            output_dir = '/kaggle/working/sex_prediction_data'
        
        # Try to find models in the working directory or input directories
        model_search_paths = ['/kaggle/working', '/kaggle/input']
    else:
        # Local environment
        if output_dir is None:
            output_dir = os.path.join('hapmap_data', 'sex_prediction_data')
        
        # Try to find models in the current directory structure
        model_search_paths = ['.', 'hapmap_data']
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Define model file paths
    best_model_file = os.path.join(output_dir, 'best_sex_model.pkl')
    selector_file = os.path.join(output_dir, 'feature_selector.pkl')
    pca_file = os.path.join(output_dir, 'pca_model.pkl')
    
    # Check if models exist in the specified directory
    models_exist = os.path.exists(best_model_file) and os.path.exists(selector_file) and os.path.exists(pca_file)
    
    # If models don't exist, try to find them
    if not models_exist:
        print("Models not found in specified directory. Searching for models...")
        
        for search_path in model_search_paths:
            # Look for model files recursively
            for model_name in ['best_sex_model.pkl', 'feature_selector.pkl', 'pca_model.pkl']:
                found_files = glob.glob(os.path.join(search_path, '**', model_name), recursive=True)
                
                if found_files:
                    # Use the first found file
                    found_file = found_files[0]
                    target_file = os.path.join(output_dir, model_name)
                    
                    # Copy or load and save the model
                    if not os.path.exists(target_file):
                        try:
                            # Try to copy the file
                            import shutil
                            shutil.copy2(found_file, target_file)
                            print(f"Copied {model_name} from {found_file} to {target_file}")
                        except Exception as e:
                            # If copy fails, load and save
                            model = joblib.load(found_file)
                            joblib.dump(model, target_file)
                            print(f"Loaded and saved {model_name} from {found_file} to {target_file}")
    
    # Check again if all models exist
    models_exist = os.path.exists(best_model_file) and os.path.exists(selector_file) and os.path.exists(pca_file)
    
    if not models_exist:
        missing_files = []
        if not os.path.exists(best_model_file):
            missing_files.append('best_sex_model.pkl')
        if not os.path.exists(selector_file):
            missing_files.append('feature_selector.pkl')
        if not os.path.exists(pca_file):
            missing_files.append('pca_model.pkl')
        
        raise FileNotFoundError(f"Required model files not found: {', '.join(missing_files)}. "
                               f"Please place these files in {output_dir}.")
    
    # Load the models
    print("Loading models...")
    best_model = joblib.load(best_model_file)
    selector = joblib.load(selector_file)
    pca = joblib.load(pca_file)
    
    # Try to find MAP file if needed
    map_file = os.path.join('hapmap_data', 'hapmap3_r3_b36_fwd.consensus.qc.poly.map')
    if not os.path.exists(map_file):
        # Search for a MAP file
        for search_path in model_search_paths:
            map_files = glob.glob(os.path.join(search_path, '**', '*.map'), recursive=True)
            if map_files:
                map_file = map_files[0]
                print(f"Found MAP file: {map_file}")
                break
    
    # Read MAP file if it exists
    if os.path.exists(map_file):
        print(f"Reading map file: {map_file}")
        map_columns = ['CHR', 'SNP', 'GEN_DIST', 'POS']
        map_df = pd.read_csv(map_file, sep='\\s+', header=None, names=map_columns)
        
        if map_df.shape[0] > max_snps:
            print(f"Using first {max_snps} SNPs out of {map_df.shape[0]} to speed up processing...")
            map_df = map_df.iloc[:max_snps]
    else:
        # Try to use the selected SNPs file
        selected_snps_file = os.path.join(output_dir, 'sex_selected_snps.csv')
        if os.path.exists(selected_snps_file):
            print(f"Using selected SNPs file: {selected_snps_file}")
            map_df = pd.read_csv(selected_snps_file)
        else:
            print("WARNING: No MAP file or selected SNPs file found. Processing may fail if SNP information is required.")
            map_df = None
    
    # Process the input file
    print(f"Processing input file: {input_file}")
    
    if not csv_input:
        # PED file processing
        # Extract the file basename to determine population
        file_basename = os.path.basename(input_file)
        population = file_basename.split('.')[1] if len(file_basename.split('.')) > 2 else "Unknown"
        
        # Read sample information (first 6 columns of PED file)
        ped_columns = ['FID', 'IID', 'PAT', 'MAT', 'SEX', 'PHENO']
        dtype_dict = {0: str, 1: str, 2: str, 3: str, 4: int, 5: int}
        
        ped_info_df = pd.read_csv(input_file, sep='\\s+', header=None, usecols=range(6), 
                                 names=ped_columns, dtype=dtype_dict, low_memory=False)
        
        # Filter to valid sex values
        valid_sex_mask = ped_info_df['SEX'].isin([1, 2])  # 1=male, 2=female
        ped_info_df = ped_info_df[valid_sex_mask]
        
        if ped_info_df.empty:
            raise ValueError("No valid samples found with sex information (1=male, 2=female).")
        
        print(f"Found {ped_info_df.shape[0]} samples with valid sex information.")
        print(f"Sex distribution: Males={sum(ped_info_df['SEX'] == 1)}, Females={sum(ped_info_df['SEX'] == 2)}")
        
        # Add population information
        ped_info_df['Population'] = population
        
        # Read the genotype data
        total_columns = 6 + (max_snps * 2)
        column_indices = list(range(6)) + list(range(6, 6 + (max_snps * 2)))
        
        try:
            genotype_data = pd.read_csv(input_file, sep='\\s+', header=None, usecols=column_indices, low_memory=False)
        except ValueError as e:
            # The file might not have enough columns
            print(f"Error reading full genotype data: {str(e)}")
            print("Trying to read available columns...")
            
            # Count columns in the file
            with open(input_file, 'r') as f:
                first_line = f.readline().strip()
                column_count = len(first_line.split())
            
            # Adjust max_snps based on available columns
            available_snp_columns = column_count - 6
            max_available_snps = available_snp_columns // 2
            
            if max_available_snps < 1000:
                raise ValueError(f"File contains only {max_available_snps} SNPs, which is insufficient for analysis. Minimum required is 1000.")
            
            print(f"File contains {max_available_snps} SNPs. Adjusting max_snps from {max_snps} to {max_available_snps}.")
            max_snps = max_available_snps
            
            # Re-read with adjusted columns
            column_indices = list(range(6)) + list(range(6, column_count))
            genotype_data = pd.read_csv(input_file, sep='\\s+', header=None, usecols=column_indices, low_memory=False)
        
        # Filter to valid samples
        genotype_data = genotype_data[valid_sex_mask].reset_index(drop=True)
        
        # Extract SNP data and encode
        snp_data = genotype_data.iloc[:, 6:].values
        encoded_data = improved_encoding(snp_data)
    
    else:
        # CSV file processing
        try:
            # Try to read the CSV file and determine its structure
            csv_df = pd.read_csv(input_file)
            print(f"CSV file loaded. Shape: {csv_df.shape}")
            
            # Check if this is a patient file with known format (based on file naming)
            file_basename = os.path.basename(input_file)
            
            # Extract information from filename if it matches expected pattern
            # Example: NA18976_JPT_Female.csv
            parts = file_basename.replace('.csv', '').split('_')
            
            if len(parts) >= 3:
                sample_id = parts[0]
                population = parts[1]
                sex_label = parts[2].lower()
                
                # Determine sex value (1=male, 2=female)
                sex_value = 2 if sex_label == 'female' else 1 if sex_label == 'male' else None
                
                if sex_value is None:
                    print(f"WARNING: Could not determine sex from filename '{file_basename}'")
                    print("Assuming this is a sample with unknown sex to be predicted.")
                    # Create a placeholder with unknown sex
                    ped_info_df = pd.DataFrame({
                        'FID': [sample_id],
                        'IID': [sample_id],
                        'PAT': ['0'],
                        'MAT': ['0'],
                        'SEX': [0],  # Unknown sex
                        'PHENO': [-9],
                        'Population': [population]
                    })
                else:
                    print(f"Detected sample: ID={sample_id}, Population={population}, Sex={sex_label} ({sex_value})")
                    ped_info_df = pd.DataFrame({
                        'FID': [sample_id],
                        'IID': [sample_id],
                        'PAT': ['0'],
                        'MAT': ['0'],
                        'SEX': [sex_value],
                        'PHENO': [-9],
                        'Population': [population]
                    })
            else:
                # If filename doesn't match expected pattern, check for columns in the CSV
                if 'IID' in csv_df.columns and 'SEX' in csv_df.columns:
                    print("Found IID and SEX columns in CSV file.")
                    ped_info_df = csv_df[['IID', 'SEX']].copy()
                    if 'FID' not in ped_info_df.columns:
                        ped_info_df['FID'] = ped_info_df['IID']
                    if 'PAT' not in ped_info_df.columns:
                        ped_info_df['PAT'] = '0'
                    if 'MAT' not in ped_info_df.columns:
                        ped_info_df['MAT'] = '0'
                    if 'PHENO' not in ped_info_df.columns:
                        ped_info_df['PHENO'] = -9
                    if 'Population' not in ped_info_df.columns:
                        ped_info_df['Population'] = 'Unknown'
                else:
                    # Create a minimal info dataframe
                    print("CSV file format not recognized. Creating a placeholder sample.")
                    ped_info_df = pd.DataFrame({
                        'FID': ['SAMPLE1'],
                        'IID': ['SAMPLE1'],
                        'PAT': ['0'],
                        'MAT': ['0'],
                        'SEX': [0],  # Unknown sex
                        'PHENO': [-9],
                        'Population': ['Unknown']
                    })
            
            # Now we need to process the SNP data
            # This requires knowing which columns contain SNP data and in what format
            
            # Check if CSV already has PC columns (already processed data)
            pc_columns = [col for col in csv_df.columns if col.startswith('PC_')]
            if pc_columns:
                print(f"Found {len(pc_columns)} principal component columns. Using these directly.")
                X_pca = csv_df[pc_columns].values
                
                # If sex is in the CSV, use it
                if 'SEX' in csv_df.columns:
                    ped_info_df['SEX'] = csv_df['SEX']
                
                # Skip the feature selection and PCA steps
                return_early = True
                
            else:
                # Check for SNP columns (rs... or similar)
                snp_columns = [col for col in csv_df.columns if col.startswith('rs') or col.startswith('SNP')]
                
                if snp_columns:
                    print(f"Found {len(snp_columns)} SNP columns in CSV file.")
                    # Extract SNP data directly (already encoded)
                    encoded_data = csv_df[snp_columns].values
                    
                    if encoded_data.shape[1] < 1000:
                        print(f"WARNING: Only {encoded_data.shape[1]} SNPs found, which may be insufficient for reliable prediction.")
                    
                else:
                    # Last resort: assume all numeric columns except standard ones are SNP data
                    non_snp_cols = ['FID', 'IID', 'PAT', 'MAT', 'SEX', 'PHENO', 'Population'] + pc_columns
                    potential_snp_cols = [col for col in csv_df.columns if col not in non_snp_cols]
                    
                    # Check if these are numeric
                    numeric_cols = [col for col in potential_snp_cols if pd.api.types.is_numeric_dtype(csv_df[col])]
                    
                    if numeric_cols:
                        print(f"Using {len(numeric_cols)} numeric columns as potential SNP data.")
                        encoded_data = csv_df[numeric_cols].values
                    else:
                        raise ValueError("Could not identify SNP data columns in the CSV file.")
                
                return_early = False
        
        except Exception as e:
            raise ValueError(f"Error processing CSV file: {str(e)}")
    
    # At this point, both PED and CSV processing should have:
    # 1. ped_info_df - containing sample info with SEX values
    # 2. encoded_data - containing the encoded SNP data
    
    if csv_input and return_early:
        features_df = pd.DataFrame(X_pca, columns=[f'PC_{i+1}' for i in range(X_pca.shape[1])])
        features_df['IID'] = ped_info_df['IID'].values
        features_df['SEX'] = ped_info_df['SEX'].values
        features_df['Population'] = ped_info_df['Population'].values
    
    print(f"Encoded {encoded_data.shape[0]} samples and {encoded_data.shape[1]} SNPs.")
    
    # Clean up to save memory
    del genotype_data, snp_data
    gc.collect()
    
    # Apply feature selection
    print("Applying feature selection...")
    X = encoded_data
    y = ped_info_df['SEX'].values
    
    try:
        X_selected = selector.transform(X)
        print(f"Selected {X_selected.shape[1]} SNPs out of {X.shape[1]} SNPs.")
    except ValueError as e:
        print(f"Error in feature selection: {str(e)}")
        print("This may be due to a mismatch in the number of SNPs between the training data and this file.")
        print("Attempting to train a new feature selector...")
        
        # Create a new selector trained on this data
        new_selector = SelectKBest(f_classif, k=min(10000, X.shape[1] // 2))
        X_selected = new_selector.fit_transform(X, y)
        print(f"Created new feature selector. Selected {X_selected.shape[1]} SNPs out of {X.shape[1]} SNPs.")
    
    # Apply PCA
    print("Applying PCA transformation...")
    try:
        X_pca = pca.transform(X_selected)
        print(f"Reduced dimensions to {X_pca.shape[1]} principal components.")
    except ValueError as e:
        print(f"Error in PCA transformation: {str(e)}")
        print("Creating a new PCA model...")
        
        # Create a new PCA model
        new_pca = PCA(n_components=min(50, X_selected.shape[1]))
        X_pca = new_pca.fit_transform(X_selected)
        print(f"Created new PCA model. Reduced dimensions to {X_pca.shape[1]} principal components.")
    
    # Create features dataframe
    features_df = pd.DataFrame(X_pca, columns=[f'PC_{i+1}' for i in range(X_pca.shape[1])])
    features_df['IID'] = ped_info_df['IID'].values
    features_df['SEX'] = ped_info_df['SEX'].values
    features_df['Population'] = ped_info_df['Population'].values
    
    # Prepare for prediction
    feature_columns = [f'PC_{i+1}' for i in range(X_pca.shape[1])]
    
    # Add population encoding if needed (if the model was trained with it)
    le_pop = LabelEncoder()
    features_df['Population_encoded'] = le_pop.fit_transform(features_df['Population'])
    feature_columns.append('Population_encoded')
    
    # Make predictions
    print("Making predictions...")
    try:
        X_pred = features_df[feature_columns].values
        y_pred = best_model.predict(X_pred)
    except Exception as e:
        print(f"Error during prediction: {str(e)}")
        print("This could be due to a mismatch in features. Trying with only PCA components...")
        
        feature_columns = [f'PC_{i+1}' for i in range(X_pca.shape[1])]
        X_pred = features_df[feature_columns].values
        
        try:
            y_pred = best_model.predict(X_pred)
        except Exception as e2:
            print(f"Second attempt failed: {str(e2)}")
            print("Using a basic RandomForest classifier as a fallback...")
            
            from sklearn.ensemble import RandomForestClassifier
            fallback_model = RandomForestClassifier(n_estimators=100, random_state=42)
            fallback_model.fit(X_pred, y)
            y_pred = fallback_model.predict(X_pred)
            print("Fallback model trained and predictions made. Note: These are not from the pre-trained model!")
    
    # Add predictions to dataframe
    features_df['Predicted_SEX'] = y_pred
    
    # Calculate accuracy
    accuracy = accuracy_score(features_df['SEX'], features_df['Predicted_SEX'])
    print(f"\nOverall accuracy: {accuracy:.4f}")
    
    # Print classification report
    print("\nClassification Report:")
    print(classification_report(features_df['SEX'], features_df['Predicted_SEX'], 
                              target_names=['Male (1)', 'Female (2)']))
    
    # Create confusion matrix
    cm = confusion_matrix(features_df['SEX'], features_df['Predicted_SEX'])
    
    if visualize:
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=['Male', 'Female'],
                   yticklabels=['Male', 'Female'])
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.title('Confusion Matrix for Sex Classification')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'new_prediction_confusion_matrix.png'))
        plt.show()
        
        # Create PCA visualization
        if X_pca.shape[1] >= 2:
            plt.figure(figsize=(10, 8))
            sex_colors = {1: 'blue', 2: 'red'}  # 1=male, 2=female
            sex_labels = {1: 'Male', 2: 'Female'}
            
            for sex_value in [1, 2]:
                sex_mask = features_df['SEX'] == sex_value
                plt.scatter(
                    features_df.loc[sex_mask, 'PC_1'], 
                    features_df.loc[sex_mask, 'PC_2'],
                    c=sex_colors[sex_value],
                    label=sex_labels[sex_value], 
                    alpha=0.7
                )
            
            plt.xlabel('Principal Component 1')
            plt.ylabel('Principal Component 2')
            plt.title('Sex Distribution in PCA Space')
            plt.legend()
            
            plt.savefig(os.path.join(output_dir, 'new_prediction_pca.png'))
            plt.show()
    
    # Save results
    results_file = os.path.join(output_dir, 'new_sex_predictions.csv')
    features_df.to_csv(results_file, index=False)
    print(f"Results saved to: {results_file}")
    
    return features_df

# Example usage for Kaggle environment
if __name__ == "__main__":
    # Detect environment
    in_kaggle = os.path.exists('/kaggle')
    if in_kaggle:
        # Search in multiple possible locations in Kaggle
        search_dirs = [
            '/kaggle/input',
            '/kaggle/working',
            '/kaggle/working/hapmap_data',
            '/kaggle/working/hapmap_data/hapmap3_r2_b36_fwd.qc.poly'
        ]
        output_dir = '/kaggle/working/sex_prediction_data'
    else:
        print("Running in local environment")
        search_dirs = ['.', 'hapmap_data']
        output_dir = 'hapmap_data/sex_prediction_data'
    
    # Try to find PED files in all search directories
    ped_files = []
    for dir_path in search_dirs:
        if os.path.exists(dir_path):
            found_files = glob.glob(os.path.join(dir_path, '**/*.ped'), recursive=True)
            ped_files.extend(found_files)
    
    # Remove duplicates
    ped_files = list(set(ped_files))
    
    # Also check for CSV files that might contain SNP data
    patient_files_dir = os.path.join('/kaggle/working', 'patient_files_20250414_124034')
    csv_files = []
    if os.path.exists(patient_files_dir):
        csv_files = glob.glob(os.path.join(patient_files_dir, '*.csv'))

    if ped_files:
        print(f"Found {len(ped_files)} PED files.")
        
        # Sort the files to make selection more predictable
        ped_files.sort()
        
        
        # Default to using the first file
        input_file = ped_files[0]
        
        # Allow for selection if there are multiple files
        if len(ped_files) > 1:
            try:
                selected_idx = 1
                if 0 <= selected_idx < len(ped_files):
                    input_file = ped_files[selected_idx]
            except ValueError:
                print("Invalid selection, using the first file.")
        
        
        # Run prediction on PED file
        try:
            results = predict_sex_from_dna(
                input_file=input_file,
                output_dir=output_dir,
                max_snps=300000,  # Use up to 300,000 SNPs
                visualize=True    # Create visualization plots
            )
            
            # Display the first few results
            print("\nSample predictions:")
            print(results[['IID', 'SEX', 'Predicted_SEX']].head(10))
            
            # Print accuracy by population
            if 'Population' in results.columns:
                print("\nAccuracy by population:")
                for pop in results['Population'].unique():
                    pop_df = results[results['Population'] == pop]
                    pop_acc = accuracy_score(pop_df['SEX'], pop_df['Predicted_SEX'])
                    print(f"  {pop}: {pop_acc:.4f} ({len(pop_df)} samples)")
            
        except Exception as e:
            print(f"Error during prediction: {str(e)}")
    elif csv_files:
        print("\nNo PED files found, but CSV files are available.")
        print("\nAvailable CSV files:")
        for i, file_path in enumerate(csv_files):
            print(f"[{i}] {file_path}")
        
        # Default to using the first file
        input_file = csv_files[0]
        
        # Allow for selection if there are multiple files
        if len(csv_files) > 1:
            try:
                selected_idx = int(input("Enter the number of the CSV file to use (or press Enter for default): ") or 0)
                if 0 <= selected_idx < len(csv_files):
                    input_file = csv_files[selected_idx]
            except ValueError:
                print("Invalid selection, using the first file.")
        
        print(f"\nUsing CSV file: {input_file}")
        print("NOTE: Support for direct CSV input is limited and depends on the CSV format matching expected structure.")
        
        # Run prediction on CSV file
        try:
            results = predict_sex_from_dna(
                input_file=input_file,
                output_dir=output_dir,
                max_snps=300000,  # Use up to 300,000 SNPs
                visualize=True,   # Create visualization plots
                csv_input=True    # Use CSV input mode
            )
        except Exception as e:
            print(f"Error processing CSV file: {str(e)}")
            print("The CSV format may not be compatible with this analysis pipeline.")
            print("Please convert your data to PED format or provide a sample of the CSV format for further assistance.")
    else:
        print(f"No PED or CSV files found in the searched directories. Please upload a PED file or specify a valid path.")
        print("\nSearched directories:")
        for dir_path in search_dirs:
            print(f"- {dir_path}")

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

class SexPredictor:
    """
    Class for predicting sex (male/female) using the trained model
    """
    def __init__(self, model_path, features_path=None, pca_path=None, selector_path=None, selected_snps_path=None):
        """
        Initialize the class with the required file paths
        
        Parameters:
        -----------
        model_path : str
            Path to the trained model file
        features_path : str, optional
            Path to the features data file
        pca_path : str, optional
            Path to the PCA model file
        selector_path : str, optional
            Path to the feature selector file
        selected_snps_path : str, optional
            Path to the selected SNPs file
        """
        print("Loading the sex prediction model...")
        
        # Load the model
        self.model = joblib.load(model_path)
        
        # Load the features data used for training (if available)
        self.features_df = None
        if features_path and os.path.exists(features_path):
            self.features_df = pd.read_csv(features_path)
        
        # Load the PCA model (if available)
        self.pca = None
        if pca_path and os.path.exists(pca_path):
            self.pca = joblib.load(pca_path)
        
        # Load the feature selector (if available)
        self.selector = None
        if selector_path and os.path.exists(selector_path):
            self.selector = joblib.load(selector_path)
        
        # Load the selected SNPs (if available)
        self.selected_snps = None
        if selected_snps_path and os.path.exists(selected_snps_path):
            self.selected_snps = pd.read_csv(selected_snps_path)
        
        # Sex labels for display
        self.sex_labels = {1: 'Male', 2: 'Female'}
        
        # Number of principal components (from training data if available)
        self.n_components = 0
        if self.features_df is not None:
            pc_columns = [col for col in self.features_df.columns if col.startswith('PC_')]
            self.n_components = len(pc_columns)
        
        print(f"Model loaded successfully.")
        if self.features_df is not None:
            print(f"Number of samples in feature data: {len(self.features_df)}")
            print(f"Number of principal components: {self.n_components}")
        if self.selected_snps is not None:
            print(f"Number of selected SNPs: {len(self.selected_snps)}")
    
    def predict_by_id(self, sample_id):
        """
        Predict sex using the sample ID present in the training data
        
        Parameters:
        -----------
        sample_id : str
            The ID of the sample to predict
            
        Returns:
        --------
        tuple
            (predicted_sex_code, predicted_sex_label, true_sex_code, true_sex_label)
        """
        if self.features_df is None:
            print("Error: No feature data available for prediction by ID.")
            return None, None, None, None
            
        if sample_id not in self.features_df['IID'].values:
            print(f"Error: Sample {sample_id} not found in the data.")
            return None, None, None, None
        
        # Extract data and make prediction
        sample_data = self.features_df[self.features_df['IID'] == sample_id]
        
        # Prepare features
        feature_cols = [col for col in sample_data.columns if col.startswith('PC_')]
        if 'Population_encoded' in sample_data.columns:
            feature_cols.append('Population_encoded')
        
                    # Make prediction
        X = sample_data[feature_cols].values
        try:
            prediction = self.model.predict(X)[0]
            predicted_sex_label = self.sex_labels.get(prediction, f"Unknown ({prediction})")
        except ValueError as e:
            if "has" in str(e) and "features" in str(e) and "expecting" in str(e):
                print(f"Error: Feature mismatch when predicting. {str(e)}")
                print(f"Current features: {feature_cols} ({len(feature_cols)} features)")
                print("This might be because the model was trained with different features.")
                print("Attempting to predict using just the PCA components...")
                
                # Try using only PCA components (without Population_encoded)
                pc_only_cols = [col for col in sample_data.columns if col.startswith('PC_')]
                X = sample_data[pc_only_cols].values
                prediction = self.model.predict(X)[0]
                predicted_sex_label = self.sex_labels.get(prediction, f"Unknown ({prediction})")
            else:
                raise e
        
        # True sex (if available)
        true_sex = None
        true_sex_label = None
        if 'SEX' in sample_data.columns:
            true_sex = sample_data['SEX'].values[0]
            true_sex_label = self.sex_labels.get(true_sex, f"Unknown ({true_sex})")
        
        return prediction, predicted_sex_label, true_sex, true_sex_label
    
    def predict_from_raw_data(self, genotype_data):
        """
        Predict sex from raw genotype data
        
        Parameters:
        -----------
        genotype_data : numpy.ndarray
            Raw genotype data (encoded)
            
        Returns:
        --------
        tuple
            (predicted_sex_code, predicted_sex_label)
        """
        if self.selector is None or self.pca is None:
            print("Error: Feature selector or PCA model not available for raw data prediction.")
            return None, None
        
        try:
            # Apply feature selection
            X_selected = self.selector.transform(genotype_data)
            
            # Apply PCA transformation
            X_pca = self.pca.transform(X_selected)
            
            # Make prediction
            prediction = self.model.predict(X_pca)[0]
            predicted_sex_label = self.sex_labels.get(prediction, f"Unknown ({prediction})")
            
            return prediction, predicted_sex_label
            
        except Exception as e:
            print(f"Error during prediction from raw data: {str(e)}")
            return None, None
    
    def display_sample_prediction(self, sample_id):
        """
        Display prediction result for a specific sample
        
        Parameters:
        -----------
        sample_id : str
            The ID of the sample to predict
        """
        predicted_sex, predicted_label, true_sex, true_label = self.predict_by_id(sample_id)
        
        if predicted_sex is None:
            return
        
        print(f"\nSample: {sample_id}")
        print(f"Predicted sex: {predicted_label} (code: {predicted_sex})")
        
        if true_sex is not None:
            print(f"True sex: {true_label} (code: {true_sex})")
            is_correct = predicted_sex == true_sex
            print(f"Prediction is {'Correct ✓' if is_correct else 'Incorrect ✗'}")
            
            # If we have confidence scores (predict_proba), display them
            if hasattr(self.model, 'predict_proba'):
                # Check if 'Population_encoded' might be causing the mismatch
                has_pop_encoded = 'Population_encoded' in self.features_df.columns
                pc_cols = [col for col in self.features_df.columns if col.startswith('PC_')]
                
                if has_pop_encoded:
                    # Likely a feature mismatch, print a helpful message
                    print("Done!")
                else:
                    try:
                        # Get only PCA columns
                        sample_features = self.features_df[self.features_df['IID'] == sample_id][pc_cols].values
                        
                        # Get prediction probabilities - handle pipeline objects differently
                        if hasattr(self.model, 'named_steps') and 'model' in self.model.named_steps:
                            # For pipelines, extract the final classifier
                            classifier = self.model.named_steps['model']
                            
                            # For ensemble models, skip confidence calculation
                            if hasattr(classifier, 'estimators_'):
                                print("Note: Confidence scores not available for ensemble models when using feature subset")
                            else:
                                # For simple classifiers in a pipeline
                                print("Note: Cannot reliably calculate confidence with pipeline models and feature mismatch")
                        else:
                            # Not a pipeline, try direct prediction
                            proba = self.model.predict_proba(sample_features)[0]
                            
                            # Find the index that corresponds to the predicted class
                            pred_idx = 0 if predicted_sex == 1 else 1
                            confidence = proba[pred_idx]
                            print(f"Confidence: {confidence:.4f} ({confidence*100:.2f}%)")
                            
                    except Exception as e:
                        print("Cannot calculate confidence score due to model structure or feature mismatch")
                    
    
    def get_available_samples(self, limit=10):
        """
        Display available sample IDs from the data
        
        Parameters:
        -----------
        limit : int, optional
            Maximum number of samples to display
            
        Returns:
        --------
        list
            List of available sample IDs
        """
        if self.features_df is None:
            print("No feature data available.")
            return []
            
        available_ids = self.features_df['IID'].unique().tolist()
        print(f"Number of available samples: {len(available_ids)}")
        print(f"Examples of available sample IDs:")
        
        for i, sample_id in enumerate(available_ids[:limit]):
            sex = "Unknown"
            pop = "Unknown"
            
            if 'SEX' in self.features_df.columns:
                sex_code = self.features_df[self.features_df['IID'] == sample_id]['SEX'].values[0]
                sex = self.sex_labels.get(sex_code, f"Unknown ({sex_code})")
                
            if 'Population' in self.features_df.columns:
                pop = self.features_df[self.features_df['IID'] == sample_id]['Population'].values[0]
                
            print(f"{i+1}. {sample_id} ({sex}, {pop})")
        
        return available_ids
    
    def analyze_prediction_accuracy(self):
        """
        Analyze and display prediction accuracy statistics
        """
        if self.features_df is None or 'SEX' not in self.features_df.columns:
            print("No feature data or sex labels available for accuracy analysis.")
            return
        
        print("\n=== Sex Prediction Accuracy Analysis ===")
        
        # Predict sex for all samples
        feature_cols = [col for col in self.features_df.columns if col.startswith('PC_')]
        if 'Population_encoded' in self.features_df.columns:
            feature_cols.append('Population_encoded')
        
        try:
            X = self.features_df[feature_cols].values
            predictions = self.model.predict(X)
        except ValueError as e:
            if "has" in str(e) and "features" in str(e) and "expecting" in str(e):
                print(f"Error in analyze_prediction_accuracy: {str(e)}")
                print("Using only PCA components for prediction...")
                
                # Use only PCA components
                pc_cols = [col for col in self.features_df.columns if col.startswith('PC_')]
                X = self.features_df[pc_cols].values
                predictions = self.model.predict(X)
            else:
                raise e
        
        # Calculate overall accuracy
        true_values = self.features_df['SEX'].values
        accuracy = np.mean(predictions == true_values)
        print(f"Overall accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
        
        # Calculate accuracy by sex
        for sex_code, sex_label in self.sex_labels.items():
            mask = true_values == sex_code
            if np.sum(mask) > 0:
                sex_accuracy = np.mean(predictions[mask] == true_values[mask])
                print(f"{sex_label} accuracy: {sex_accuracy:.4f} ({sex_accuracy*100:.2f}%) - {np.sum(mask)} samples")
        
        # Calculate accuracy by population (if available)
        if 'Population' in self.features_df.columns:
            print("\nAccuracy by population:")
            populations = self.features_df['Population'].unique()
            
            # Sort populations by size (descending)
            pop_counts = self.features_df['Population'].value_counts()
            populations = pop_counts.index.tolist()
            
            for pop in populations:
                mask = self.features_df['Population'] == pop
                if np.sum(mask) >= 5:  # Only consider populations with at least 5 samples
                    pop_true = self.features_df.loc[mask, 'SEX'].values
                    pop_pred = predictions[mask]
                    pop_accuracy = np.mean(pop_pred == pop_true)
                    print(f"{pop}: {pop_accuracy:.4f} ({pop_accuracy*100:.2f}%) - {np.sum(mask)} samples")
    
    def visualize_predictions(self, output_dir=None):
        """
        Visualize predictions using PCA components
        
        Parameters:
        -----------
        output_dir : str, optional
            Directory to save the visualization plots
        """
        if self.features_df is None or 'SEX' not in self.features_df.columns:
            print("No feature data or sex labels available for visualization.")
            return
            
        # Create output directory if specified and doesn't exist
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)
        
        # Extract data
        pc_columns = [col for col in self.features_df.columns if col.startswith('PC_')]
        if len(pc_columns) < 2:
            print("Not enough principal components for visualization.")
            return
            
        # Make predictions
        try:
            feature_cols = pc_columns.copy()
            if 'Population_encoded' in self.features_df.columns:
                feature_cols.append('Population_encoded')
                
            X = self.features_df[feature_cols].values
            predictions = self.model.predict(X)
        except ValueError as e:
            if "has" in str(e) and "features" in str(e) and "expecting" in str(e):
                print(f"Error in visualize_predictions: {str(e)}")
                print("Using only PCA components for prediction...")
                
                # Use only PCA components
                X = self.features_df[pc_columns].values
                predictions = self.model.predict(X)
            else:
                raise e
        
        # Add predictions to the dataframe
        self.features_df['Predicted_SEX'] = predictions
        
        # Create a column to indicate correct/incorrect predictions
        self.features_df['Correct'] = self.features_df['SEX'] == self.features_df['Predicted_SEX']
        
        # Create PCA visualization (first two components)
        plt.figure(figsize=(12, 10))
        
        # Define colors and markers
        colors = {1: 'blue', 2: 'red'}
        markers = {True: 'o', False: 'x'}
        
        # Plot each sample
        for sex in [1, 2]:
            for correct in [True, False]:
                mask = (self.features_df['SEX'] == sex) & (self.features_df['Correct'] == correct)
                if np.sum(mask) > 0:
                    plt.scatter(
                        self.features_df.loc[mask, pc_columns[0]],
                        self.features_df.loc[mask, pc_columns[1]],
                        c=colors[sex],
                        marker=markers[correct],
                        alpha=0.7,
                        label=f"{self.sex_labels[sex]} ({'Correct' if correct else 'Incorrect'})"
                    )
        
        plt.xlabel(pc_columns[0])
        plt.ylabel(pc_columns[1])
        plt.title('Sex Classification Results in PCA Space')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        if output_dir:
            plt.savefig(os.path.join(output_dir, 'sex_classification_results.png'), dpi=300)
            print(f"Visualization saved to: {os.path.join(output_dir, 'sex_classification_results.png')}")
        else:
            plt.show()
        
        # If population data is available, create visualization by population
        if 'Population' in self.features_df.columns:
            top_pops = self.features_df['Population'].value_counts().head(9).index.tolist()
            
            plt.figure(figsize=(15, 12))
            
            for i, pop in enumerate(top_pops):
                plt.subplot(3, 3, i+1)
                
                pop_mask = self.features_df['Population'] == pop
                pop_data = self.features_df[pop_mask]
                
                for sex in [1, 2]:
                    for correct in [True, False]:
                        mask = (pop_data['SEX'] == sex) & (pop_data['Correct'] == correct)
                        if np.sum(mask) > 0:
                            plt.scatter(
                                pop_data.loc[mask, pc_columns[0]],
                                pop_data.loc[mask, pc_columns[1]],
                                c=colors[sex],
                                marker=markers[correct],
                                alpha=0.7,
                                s=30
                            )
                
                # Calculate accuracy for this population
                pop_accuracy = np.mean(pop_data['Correct'])
                
                plt.title(f'Population: {pop}\nAccuracy: {pop_accuracy:.3f}')
                plt.xlabel(pc_columns[0])
                plt.ylabel(pc_columns[1])
                
            plt.tight_layout()
            
            if output_dir:
                plt.savefig(os.path.join(output_dir, 'sex_classification_by_population.png'), dpi=300)
                print(f"Population visualization saved to: {os.path.join(output_dir, 'sex_classification_by_population.png')}")
            else:
                plt.show()


# Function to check for required files
def check_required_files(model_dir):
    """
    Check for the required files in the specified directory
    
    Parameters:
    -----------
    model_dir : str
        Directory containing the model files
        
    Returns:
    --------
    bool
        True if all required files are found, False otherwise
    """
    required_files = {
        'best_sex_model.pkl': 'Sex classification model'
    }
    
    optional_files = {
        'ensemble_sex_model.pkl': 'Ensemble sex model',
        'sex_features_pca.csv': 'Feature data',
        'sex_predictions.csv': 'Prediction data',
        'sex_selected_snps.csv': 'Selected SNPs list',
        'pca_model.pkl': 'PCA model',
        'feature_selector.pkl': 'Feature selector'
    }
    
    # Check for required files
    for filename, description in required_files.items():
        filepath = os.path.join(model_dir, filename)
        if not os.path.exists(filepath):
            print(f"❌ {description} is missing: {filepath}")
            return False
        else:
            print(f"✓ {description} is present: {filepath}")
    
    # Check for optional files
    for filename, description in optional_files.items():
        filepath = os.path.join(model_dir, filename)
        if not os.path.exists(filepath):
            print(f"ℹ️ {description} is missing: {filepath}")
        else:
            print(f"✓ {description} is present: {filepath}")
    
    return True


# Function for improved encoding of genetic data
def improved_encoding(genotype_data):
    """
    Improved encoding for genetic data using additive inheritance model
    
    Parameters:
    -----------
    genotype_data : numpy.ndarray
        Raw genotype data
        
    Returns:
    --------
    numpy.ndarray
        Encoded genetic data
    """
    encoded_data = np.zeros((genotype_data.shape[0], genotype_data.shape[1] // 2), dtype=np.float32)
    
    for i in range(0, genotype_data.shape[1], 2):
        allele1 = genotype_data[:, i]
        allele2 = genotype_data[:, i+1]
        snp_idx = i // 2
        
        unique_alleles, counts = np.unique(np.concatenate([allele1, allele2]), return_counts=True)
        
        if len(unique_alleles) > 0:
            ref_allele = unique_alleles[np.argmax(counts)]
            
            alt_count = np.zeros(allele1.shape[0], dtype=np.float32)
            alt_count += (allele1 != ref_allele) & (allele1 != '0')
            alt_count += (allele2 != ref_allele) & (allele2 != '0')
            
            encoded_data[:, snp_idx] = alt_count
    
    return encoded_data


# Main function to run in Kaggle or other environment
def main():
    """
    Main function for sex prediction
    """
    print("=== Sex Prediction System from Genetic Data ===\n")
    
    # Set the model directory path
    # In Kaggle, it is better to use the absolute path to avoid path issues
    model_dir = '/kaggle/working/hapmap_data/sex_prediction_data'
    
    if not os.path.exists(model_dir):
        alternative_dirs = [
            'hapmap_data/sex_prediction_data',
            './hapmap_data/sex_prediction_data',
            './sex_prediction_data'
        ]
        
        for alt_dir in alternative_dirs:
            if os.path.exists(alt_dir):
                model_dir = alt_dir
                print(f"Model directory path has been changed to: {model_dir}")
                break
        else:
            print(f"Error: Model directory not found. Tried multiple paths.")
            print("Please make sure the model directory exists and contains the required files.")
            return
    
    # Check for required files
    if not check_required_files(model_dir):
        print("Some required files are missing. Ensure that the model has been trained first.")
        return
    
    # Load the model and make predictions
    try:
        # File paths
        model_path = os.path.join(model_dir, 'best_sex_model.pkl')
        ensemble_model_path = os.path.join(model_dir, 'ensemble_sex_model.pkl')
        features_path = os.path.join(model_dir, 'sex_features_pca.csv')
        prediction_path = os.path.join(model_dir, 'sex_predictions.csv')
        selected_snps_path = os.path.join(model_dir, 'sex_selected_snps.csv')
        pca_path = os.path.join(model_dir, 'pca_model.pkl')
        selector_path = os.path.join(model_dir, 'feature_selector.pkl')
        
        # Choose the best model available
        chosen_model_path = ensemble_model_path if os.path.exists(ensemble_model_path) else model_path
        chosen_features_path = prediction_path if os.path.exists(prediction_path) else features_path
        
        # Initialize the predictor class
        predictor = SexPredictor(
            model_path=chosen_model_path,
            features_path=chosen_features_path,
            pca_path=pca_path if os.path.exists(pca_path) else None,
            selector_path=selector_path if os.path.exists(selector_path) else None,
            selected_snps_path=selected_snps_path if os.path.exists(selected_snps_path) else None
        )
        
        # Display available samples
        available_ids = predictor.get_available_samples(limit=10)
        
        # Analyze prediction accuracy
        predictor.analyze_prediction_accuracy()
        
        # Test prediction on a sample
        if available_ids:
            print("\n=== Test Prediction ===")
            sample_id = available_ids[0]  # Use the first sample for testing
            predictor.display_sample_prediction(sample_id)
            
            # Visualize predictions
            try:
                predictor.visualize_predictions(output_dir=model_dir)
            except Exception as e:
                print(f"Error during visualization: {str(e)}")
                print("Continuing with predictions...")
            
            # Allow user to input a sample ID for prediction
            while True:
                user_input = input("\nEnter a sample ID to predict (press q to quit): ")
                if user_input.lower() == 'q':
                    break
                predictor.display_sample_prediction(user_input)
        
    except Exception as e:
        print(f"❌ An error occurred: {str(e)}")
        import traceback
        print(traceback.format_exc())


if __name__ == "__main__":
    main()

=== Sex Prediction System from Genetic Data ===

✓ Sex classification model is present: /kaggle/working/hapmap_data/sex_prediction_data/best_sex_model.pkl
✓ Ensemble sex model is present: /kaggle/working/hapmap_data/sex_prediction_data/ensemble_sex_model.pkl
✓ Feature data is present: /kaggle/working/hapmap_data/sex_prediction_data/sex_features_pca.csv
✓ Prediction data is present: /kaggle/working/hapmap_data/sex_prediction_data/sex_predictions.csv
✓ Selected SNPs list is present: /kaggle/working/hapmap_data/sex_prediction_data/sex_selected_snps.csv
✓ PCA model is present: /kaggle/working/hapmap_data/sex_prediction_data/pca_model.pkl
✓ Feature selector is present: /kaggle/working/hapmap_data/sex_prediction_data/feature_selector.pkl
Loading the sex prediction model...
Model loaded successfully.
Number of samples in feature data: 1397
Number of principal components: 50
Number of selected SNPs: 10000
Number of available samples: 1397
Examples of available sample IDs:
1. NA18597 (Female, C


Enter a sample ID to predict (press q to quit):  NA18597



Sample: NA18597
Predicted sex: Female (code: 2)
True sex: Female (code: 2)
Prediction is Correct ✓
Done!



Enter a sample ID to predict (press q to quit):  NA18557



Sample: NA18557
Predicted sex: Male (code: 1)
True sex: Male (code: 1)
Prediction is Correct ✓
Done!
